**Phase 3a : Build Knowledge Graph**
- Build knowledge graph from Silver chunks
- Trigger: After 01_silver_processing completes
- Input:   silver_llm/not_personal/email_chunks/, attachment_chunks/
- Output:  gold_llm/knowledge_graph/ on ADLS

**Enviroment variables setup**

In [ ]:
import os

os.environ["AZURE_OPENAI_ENDPOINT"] = "https://<YOUR_OPENAI_RESOURCE>.services.ai.azure.com/"
os.environ["AZURE_OPENAI_API_KEY"] = "<YOUR_OPENAI_KEY>"
os.environ["AZURE_OPENAI_DEPLOYMENT"] = "gpt-4o"
os.environ["AZURE_OPENAI_API_VERSION"] = "2025-01-01-preview"
os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT"] = "text-embedding-3-small"
os.environ["ADLS_STORAGE_ACCOUNT"] = "<YOUR_ADLS_ACCOUNT>"
os.environ["ADLS_CONTAINER"] = "<YOUR_ADLS_CONTAINER>"
os.environ["ADLS_STORAGE_KEY"] = "<YOUR_ADLS_KEY>"
os.environ["COSMOS_GREMLIN_ENDPOINT"] = "wss://<YOUR_COSMOS_ACCOUNT>.gremlin.cosmos.azure.com:443/"
os.environ["COSMOS_GREMLIN_KEY"] = "<YOUR_COSMOS_KEY>"
os.environ["COSMOS_DATABASE"] = "email_cosmos_db"
os.environ["COSMOS_GRAPH"] = "email_cosmos_graph"
os.environ["COSMOS_NOSQL_ENDPOINT"] = ""
os.environ["COSMOS_NOSQL_KEY"] = ""
os.environ["AZURE_SEARCH_ENDPOINT"] = "https://<YOUR_SEARCH_SERVICE>.search.windows.net"
os.environ["AZURE_SEARCH_API_KEY"] = "<YOUR_SEARCH_KEY>"
os.environ["AZURE_SEARCH_INDEX_NAME"] = "kg-chunks"
os.environ["AZURE_SEARCH_ENTITY_INDEX_NAME"] = "kg-entities"
os.environ["AZURE_SEARCH_COMMUNITY_INDEX_NAME"] = "kg-communities"
os.environ["PIPELINE_MODE"] = "llm"
os.environ["PROCESS_ATTACHMENTS"] = "true"
print("Environment variables set.")


**Download code from ADLS**

In [ ]:
import os
import sys
from azure.identity import DefaultAzureCredential
from azure.storage.filedatalake import DataLakeServiceClient

service = DataLakeServiceClient(
    account_url=f"https://{os.environ['ADLS_STORAGE_ACCOUNT']}.dfs.core.windows.net",
    credential=os.environ["ADLS_STORAGE_KEY"]
)

fs = service.get_file_system_client(os.environ["ADLS_CONTAINER"])

for prefix in ["code/src", "code/config"]:
    for p in fs.get_paths(path=prefix, recursive=True):
        if not p.is_directory:
            local_path = p.name.replace("code/", "", 1)
            os.makedirs(os.path.dirname(local_path), exist_ok=True)
            with open(local_path, "wb") as f:
                file_client = fs.get_file_client(p.name)
                f.write(file_client.download_file().readall())

sys.path.insert(0, ".")
src_count = len(os.listdir("src")) if os.path.exists("src") else 0
config_count = len(os.listdir("config")) if os.path.exists("config") else 0
print(f"Downloaded: {src_count} items in src/, {config_count} items in config/")

**ADLS Adapter Init**

In [ ]:
import os
import sys
import json
import logging
from pathlib import Path
from datetime import datetime

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

sys.path.insert(0, ".")

from src.storage import ADLSAdapter

adapter = ADLSAdapter()

**Download Silver from ADLS**

In [ ]:
import os

SILVER_PATH = "silver_llm"
LOCAL_SILVER = "/tmp/pipeline/silver"

files = adapter.list_files(SILVER_PATH, "*.json")
print(f"Found {len(files)} Silver files on ADLS")

for f in files:
    rel_path = f.replace(f"{SILVER_PATH}/", "", 1)
    local_file = os.path.join(LOCAL_SILVER, rel_path)
    os.makedirs(os.path.dirname(local_file), exist_ok=True)
    with open(local_file, "wb") as fh:
        fh.write(adapter.read_bytes(f))

# Verify
for root, dirs, files_list in os.walk(LOCAL_SILVER):
    if files_list:
        print(f"{root}: {len(files_list)} files")

**Check single quotes issue:**

**Prepare LOCAL_GOLD**

In [ ]:
import shutil

LOCAL_GOLD = "/tmp/pipeline/gold"
if os.path.exists(LOCAL_GOLD):
    shutil.rmtree(LOCAL_GOLD)
os.makedirs(LOCAL_GOLD, exist_ok=True)

**Build Graph**

In [ ]:
from src.gold.graph_builder import GraphBuilder

builder = GraphBuilder(silver_path="/tmp/pipeline/silver", gold_path=LOCAL_GOLD)
knowledge_graph = builder.build_from_chunks()
builder.save()
print(f"Graph: {len(knowledge_graph.nodes)} nodes, {len(knowledge_graph.edges)} edges")

**Upload graph files to ADLS**

In [ ]:
from pathlib import Path

upload_count = 0
gold_local = Path(LOCAL_GOLD)
GOLD_PATH = "gold_llm"

for local_file in gold_local.rglob("*"):
    if local_file.is_file():
        rel_path = local_file.relative_to(gold_local)
        adapter.write_bytes(f"{GOLD_PATH}/{rel_path}", local_file.read_bytes())
        upload_count += 1

print(f"Uploaded {upload_count} Gold files to ADLS")